# Previsão de Sinistros de Seguro Automotivo

## 1. Carregando os Dados

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

path_file = 'https://raw.githubusercontent.com/MathMachado/DSWP/refs/heads/master/Dataframes/Car_Insurance_Claim.csv'

df = pd.read_csv(path_file)
df.head()

## 2. Pré-processamento dos Dados

### 2.1. Lowercase e Remoção de Caracteres Especiais

In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace('[^\w\s]', '', regex=True)
print(df.columns)
df.head()

### 2.2. Converter a coluna 'age' para numérica

In [ ]:
def age_to_numeric(age):
    if age == '16-25':
        return 20.5
    elif age == '26-39':
        return 32.5
    elif age == '40-64':
        return 52
    elif age == '65+':
        return 65
    else:
        return np.nan

df['age'] = df['age'].apply(age_to_numeric)

### 2.3. Tratamento de Valores Ausentes

In [ ]:
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    df[col].fillna(df[col].mean(), inplace=True)
for col in df.select_dtypes(include=['object']).columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

### 2.4. Tratamento de Outliers

In [ ]:
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

## 3. Análise Exploratória de Dados (EDA)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['age'], bins=20, kde=True)
plt.title('Distribuição de Idade')
plt.xlabel('Idade')
plt.ylabel('Frequência')
plt.savefig('age_distribution.png')

### 2.5. Codificação de Variáveis Categóricas

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
plt.figure(figsize=(12, 8))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm')
plt.title('Matriz de Correlação')
plt.savefig('correlation_matrix.png')

## 4. Seleção de Features

In [ ]:
X = df.drop('outcome', axis=1)
y = df['outcome']

## 5. Modelagem

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Acurácia: {accuracy:.2f}')
print(classification_report(y_test, y_pred))
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão')
plt.xlabel('Previsto')
plt.ylabel('Verdadeiro')
plt.savefig('confusion_matrix.png')

## 6. Salvar Resultados

In [ ]:
df.to_csv('cleaned_data.csv', index=False)